In [ ]:
# Cell 1 - ACLED violence: user inputs and standardized run context
from __future__ import annotations

from pathlib import Path
import json
import pandas as pd

from wia_pipelines.config import validate_run_metadata
from wia_pipelines.hazards.violence import ViolenceRunInputs, build_violence_run_context

# USER INPUTS
ISO3 = "YEM"
AS_OF_DATE = "2025-12-31"  # YYYY-MM-DD
LOOKBACK_MONTHS = 12

# Optional ACLED event-type filter.
# Keep protests excluded by default to match legacy violence runs.
SUPPORTED_EVENT_TYPES = [
    "Battles",
    "Explosions/Remote violence",
    "Violence against civilians",
    "Riots",
    "Protests",
]
INCLUDED_EVENT_TYPES = [
    "Battles",
    "Explosions/Remote violence",
    "Violence against civilians",
    "Riots",
    "Protests",
]
# To include all supported types, set: INCLUDED_EVENT_TYPES = SUPPORTED_EVENT_TYPES

OUT_ROOT = Path("./outputs").resolve()
AS_OF_DT = pd.to_datetime(AS_OF_DATE)
WINDOW_END = AS_OF_DT
WINDOW_START = (AS_OF_DT - pd.DateOffset(months=LOOKBACK_MONTHS)) + pd.Timedelta(days=1)
WINDOW_START_DATE = WINDOW_START.date().isoformat()
WINDOW_END_DATE = WINDOW_END.date().isoformat()

# Build standardized run context and schema-compliant base metadata
RUN_CTX = build_violence_run_context(
    ViolenceRunInputs(
        iso3=ISO3,
        as_of_date=AS_OF_DATE,
        lookback_months=LOOKBACK_MONTHS,
        output_root=OUT_ROOT,
    ),
    create_dirs=True,
    write_metadata=True,
)

RUN_ID = RUN_CTX["config"].run_id
RUN_METADATA = RUN_CTX["metadata"]
RUN_METADATA_PATH = RUN_CTX["metadata_path"]

# Store violence-specific settings without mutating required run_config fields
RUN_METADATA["violence_config"] = {
    "pipeline": "violence_acled_proximity",
    "iso3": ISO3,
    "as_of_date": AS_OF_DATE,
    "window_start_date": WINDOW_START_DATE,
    "window_end_date": WINDOW_END_DATE,
    "lookback_months": LOOKBACK_MONTHS,
    "supported_event_types": SUPPORTED_EVENT_TYPES,
    "included_event_types": INCLUDED_EVENT_TYPES,
    "buffer_radii_km": {
        "Battles": 5,
        "Explosions/Remote violence": 5,
        "Violence against civilians": "2_or_5",
        "Riots": 2,
        "Protests": 1,
    },
    "notes": {
        "metric": "ACLED point events with ACLED-standard 1/2/5 km buffers",
        "exposure_rule": "any-event footprint overlap within the lookback window",
        "baseline_relative_threshold": False,
        "consecutive_days_required": False,
    },
}
RUN_METADATA["pipeline"] = "violence_acled_proximity"
RUN_METADATA.setdefault("inputs", {})

validate_run_metadata(RUN_METADATA)
RUN_METADATA_PATH.write_text(json.dumps(RUN_METADATA, indent=2), encoding="utf-8")

print("ISO3:", ISO3)
print("AS_OF_DATE:", AS_OF_DATE)
print("Window:", f"{WINDOW_START_DATE} -> {WINDOW_END_DATE}", f"(m{LOOKBACK_MONTHS})")
print("Included ACLED event types:", INCLUDED_EVENT_TYPES)
print("Run ID:", RUN_ID)

In [ ]:
# Cell 2 - Paths + directories (standardized run contract)
from pathlib import Path
import json
from datetime import datetime

# INPUT PATHS
GDB_ZIP_PATH = Path("./data/cod-ab/global_admin_boundaries_matched_latest.gdb.zip")
ADMIN2_LAYER = "admin2"
worldpop_name = f"{ISO3.lower()}_pop_2025_CN_100m_R2025A_v1.tif"
WORLDPOP_PATH = Path("./data/population") / worldpop_name

# Default ACLED CSV naming pattern. Override ACLED_CSV manually if needed.
_default_acled = Path("./data/violence") / (
    f"acled_{ISO3.lower()}_{WINDOW_START_DATE.replace('-', '')}-{WINDOW_END_DATE.replace('-', '')}.csv"
)
ACLED_CSV = _default_acled
if not ACLED_CSV.exists():
    candidates = sorted(Path("./data/violence").glob(f"acled_{ISO3.lower()}_*.csv"))
    if candidates:
        ACLED_CSV = candidates[-1]

# Canonical run directories
RUN_DIR = RUN_CTX["layout"]["base"]
DIRS = {
    k: RUN_CTX["layout"][k]
    for k in ["base", "raw", "intermediate", "rasters", "tables", "qc", "logs", "cache"]
}
for p in set(DIRS.values()):
    p.mkdir(parents=True, exist_ok=True)

# Standard output filenames
VIOL_EVENT_COUNT_TIF = DIRS["rasters"] / "violence" / f"{RUN_ID}_violence_event_count.tif"
VIOL_MASK_TIF = DIRS["rasters"] / "violence" / f"{RUN_ID}_violence_mask.tif"
VIOL_POP_AFFECTED_TIF = DIRS["rasters"] / "violence" / f"{RUN_ID}_violence_pop_affected.tif"
VIOL_POP_WEIGHTED_EVENT_COUNT_TIF = (
    DIRS["rasters"] / "violence" / f"{RUN_ID}_violence_pop_weighted_event_count.tif"
)
VIOL_MASK_TIF.parent.mkdir(parents=True, exist_ok=True)

ADM2_STATS_CSV = DIRS["tables"] / f"{RUN_ID}_adm2_stats.csv"
VIOL_FOOTPRINT_GPKG = DIRS["intermediate"] / "violence" / f"{RUN_ID}_violence_footprint.gpkg"
VIOL_FOOTPRINT_GPKG.parent.mkdir(parents=True, exist_ok=True)

QC_VIOL_DIR = DIRS["qc"] / "violence"
QC_VIOL_DIR.mkdir(parents=True, exist_ok=True)
QC_EVENTS_PNG = QC_VIOL_DIR / f"{RUN_ID}_events_points.png"
QC_FOOTPRINT_PNG = QC_VIOL_DIR / f"{RUN_ID}_footprint_preview.png"
QC_MASK_PNG = QC_VIOL_DIR / f"{RUN_ID}_mask_overlay.png"
QC_ADM2_MAP_PNG = QC_VIOL_DIR / f"{RUN_ID}_adm2_pct_affected.png"

# Progress/status paths
BUFFER_STATUS_JSON = DIRS["logs"] / "violence_buffer_status.json"
RASTER_STATUS_JSON = DIRS["logs"] / "violence_raster_status.json"
ZONAL_STATUS_JSON = DIRS["logs"] / "violence_zonal_status.json"

for pth in [WORLDPOP_PATH, ACLED_CSV, GDB_ZIP_PATH]:
    if not pth.exists():
        raise FileNotFoundError(f"Missing input: {pth.resolve()}")


def _write_run_metadata() -> None:
    validate_run_metadata(RUN_METADATA)
    RUN_METADATA_PATH.write_text(json.dumps(RUN_METADATA, indent=2), encoding="utf-8")


def update_run_metadata_artifact(kind: str, path: Path, note: str = "") -> None:
    RUN_METADATA.setdefault("artifacts", [])
    RUN_METADATA["artifacts"].append(
        {
            "timestamp": datetime.utcnow().isoformat(timespec="seconds") + "Z",
            "kind": kind,
            "path": str(path),
            "note": note,
        }
    )
    _write_run_metadata()


print("Run directory:", RUN_DIR.resolve())
print("WorldPop:", WORLDPOP_PATH.resolve())
print("ACLED CSV:", ACLED_CSV.resolve())
print("COD GDB zip:", GDB_ZIP_PATH.resolve())

RUN_METADATA.setdefault("inputs", {})
RUN_METADATA["inputs"].update(
    {
        "worldpop_tif": str(WORLDPOP_PATH),
        "acled_csv": str(ACLED_CSV),
        "cod_gdb_zip": str(GDB_ZIP_PATH),
    }
)
RUN_METADATA.setdefault("paths", {})
RUN_METADATA["paths"].update({k: str(v) for k, v in DIRS.items()})
RUN_METADATA["paths"]["status_files"] = {
    "buffer": str(BUFFER_STATUS_JSON),
    "raster": str(RASTER_STATUS_JSON),
    "zonal": str(ZONAL_STATUS_JSON),
}
_write_run_metadata()
print("Wrote:", RUN_METADATA_PATH.name)

In [ ]:
# Cell 3 — Load WorldPop grid + core raster metadata (reference grid)
# Mirrors the "reference raster" pattern used in other hazard pipelines:
# - Establish CRS, transform, shape, nodata, dtype
# - Store key metadata in RUN_METADATA for reproducibility/QC

import rasterio

with rasterio.open(WORLDPOP_PATH) as src:
    WP_PROFILE = src.profile.copy()
    WP_CRS = src.crs
    WP_TRANSFORM = src.transform
    WP_HEIGHT, WP_WIDTH = src.height, src.width
    WP_NODATA = src.nodata
    WP_DTYPE = src.dtypes[0]
    WP_BOUNDS = src.bounds
    WP_RES = src.res  # (xres, yres)

print("WorldPop reference grid")
print("  CRS:", WP_CRS)
print("  Shape (H, W):", (WP_HEIGHT, WP_WIDTH))
print("  Resolution:", WP_RES)
print("  Nodata:", WP_NODATA)
print("  Dtype:", WP_DTYPE)
print("  Bounds:", WP_BOUNDS)

# Persist to metadata
RUN_METADATA["worldpop_ref"] = {
    "crs": str(WP_CRS),
    "shape": [int(WP_HEIGHT), int(WP_WIDTH)],
    "res": [float(WP_RES[0]), float(WP_RES[1])],
    "nodata": None if WP_NODATA is None else float(WP_NODATA),
    "dtype": str(WP_DTYPE),
    "bounds": [
        float(WP_BOUNDS.left),
        float(WP_BOUNDS.bottom),
        float(WP_BOUNDS.right),
        float(WP_BOUNDS.top),
    ],
}

_write_run_metadata()
print("Updated:", RUN_METADATA_PATH.name)

In [ ]:
# Cell 4 — Load + filter ACLED events (window + selected types)
# Outputs:
#   - acled_df: filtered pandas DataFrame
#   - QC: counts by event_type and by buffer_km (ACLED radii 1/2/5 km)

import pandas as pd

from wia_pipelines.hazards.violence import acled_buffer_km

# ACLED column names (expected)
LAT_COL, LON_COL = "latitude", "longitude"
DATE_COL = "event_date"
TYPE_COL = "event_type"
FATAL_COL = "fatalities"

supported_types = set(SUPPORTED_EVENT_TYPES)
selected_types = list(INCLUDED_EVENT_TYPES)
if not selected_types:
    raise ValueError("INCLUDED_EVENT_TYPES is empty. Provide at least one ACLED event type.")
unknown_types = sorted(set(selected_types) - supported_types)
if unknown_types:
    raise ValueError(
        f"INCLUDED_EVENT_TYPES contains unsupported values: {unknown_types}. "
        f"Supported: {sorted(supported_types)}"
    )
selected_type_set = set(selected_types)

raw_df = pd.read_csv(ACLED_CSV)
required = [LAT_COL, LON_COL, DATE_COL, TYPE_COL]
missing = [c for c in required if c not in raw_df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}. Found: {list(raw_df.columns)}")

# Parse and clean fields used downstream
df = raw_df.copy()
df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")
df[LAT_COL] = pd.to_numeric(df[LAT_COL], errors="coerce")
df[LON_COL] = pd.to_numeric(df[LON_COL], errors="coerce")
df = df.dropna(subset=[DATE_COL, LAT_COL, LON_COL, TYPE_COL]).copy()
df = df[df[LAT_COL].between(-90, 90) & df[LON_COL].between(-180, 180)].copy()

# Fatalities are only needed for the "Violence against civilians" radius rule
if FATAL_COL in df.columns:
    df[FATAL_COL] = pd.to_numeric(df[FATAL_COL], errors="coerce").fillna(0)
else:
    df[FATAL_COL] = 0

# Apply time window and selected-type filters
df = df[(df[DATE_COL] >= WINDOW_START) & (df[DATE_COL] <= WINDOW_END)].copy()
df = df[df[TYPE_COL].isin(selected_type_set)].copy()
if df.empty:
    raise ValueError("No ACLED events remain after filtering to window + included event types.")

# Assign ACLED-standard buffer radii (1/2/5 km)
df["buffer_km"] = [acled_buffer_km(t, f) for t, f in zip(df[TYPE_COL], df[FATAL_COL])]
acled_df = df

print(f"ACLED events after filtering: {len(acled_df):,}")
print(f"Window: {WINDOW_START_DATE} -> {WINDOW_END_DATE} (inclusive)")
print("Included event types:", selected_types)
print("\nEvent types:")
display(acled_df[TYPE_COL].value_counts())
print("\nBuffer radii (km):")
display(acled_df["buffer_km"].value_counts().sort_index())

RUN_METADATA["acled_events"] = {
    "rows_loaded": int(len(raw_df)),
    "rows_after_filter": int(len(acled_df)),
    "window_start": WINDOW_START_DATE,
    "window_end": WINDOW_END_DATE,
    "included_event_types": selected_types,
    "event_type_counts": acled_df[TYPE_COL].value_counts().to_dict(),
    "buffer_km_counts": acled_df["buffer_km"].value_counts().sort_index().to_dict(),
}
_write_run_metadata()
print("\nUpdated:", RUN_METADATA_PATH.name)

In [ ]:
# Cell 5 — Event points GeoDataFrame + preflight checks + QC map
# Outputs:
#   - events_wgs84: GeoDataFrame in EPSG:4326
#   - admin2_gdf: country admin boundaries
#   - preflight_coverage written to metadata

import geopandas as gpd
from shapely.geometry import Point
import matplotlib.pyplot as plt

from wia_pipelines.hazards.coverage_checks import check_worldpop_coverage
from wia_pipelines.core.worldpop import bbox_coverage_report

# Build events GeoDataFrame in WGS84
events_wgs84 = gpd.GeoDataFrame(
    acled_df.copy(),
    geometry=[Point(xy) for xy in zip(acled_df[LON_COL], acled_df[LAT_COL])],
    crs="EPSG:4326",
)
print("Events bounds (WGS84):", events_wgs84.total_bounds)

# Load ADM2 boundaries from zipped COD GDB and filter country
gdb_vsi_path = f"zip://{GDB_ZIP_PATH.resolve()}"
admin2_gdf = gpd.read_file(gdb_vsi_path, layer=ADMIN2_LAYER)
required_cols = {"iso3", "adm2_pcode", "geometry"}
missing_cols = required_cols - set(admin2_gdf.columns)
if missing_cols:
    raise KeyError(f"Missing required columns in COD admin2: {sorted(missing_cols)}")

admin2_gdf = admin2_gdf[admin2_gdf["iso3"] == ISO3].copy()
admin2_gdf = admin2_gdf.dropna(subset=["adm2_pcode", "geometry"]).reset_index(drop=True)
if admin2_gdf.empty:
    raise ValueError(f"No admin2 features found for ISO3={ISO3} in {GDB_ZIP_PATH} layer={ADMIN2_LAYER}")

admin2_gdf = admin2_gdf.to_crs("EPSG:4326")
admin2_gdf["geometry"] = admin2_gdf.geometry.buffer(0)
admin2_gdf = admin2_gdf[admin2_gdf.is_valid].copy()
admin2_union = admin2_gdf.geometry.union_all()

print(f"Loaded admin2 features for {ISO3}: {len(admin2_gdf):,}")
print("Admin bounds (WGS84):", admin2_gdf.total_bounds)

# Preflight checks before heavy buffering/rasterization
admin_bounds = tuple(float(v) for v in admin2_gdf.total_bounds)
wp_cov = check_worldpop_coverage(admin_bounds, WORLDPOP_PATH)
events_bounds = tuple(float(v) for v in events_wgs84.total_bounds)
events_cov = bbox_coverage_report(admin_bounds, events_bounds)
events_inside_n = int(events_wgs84.within(admin2_union).sum())

print("Preflight - WorldPop coverage %:", round(wp_cov["coverage_pct"], 3), "full=", wp_cov["full_coverage"])
print("Preflight - Event bbox coverage of admin bbox %:", round(events_cov["coverage_pct"], 3))
print("Preflight - Events inside admin boundaries:", f"{events_inside_n:,} / {len(events_wgs84):,}")

RUN_METADATA["preflight_coverage"] = {
    "thresholds": {
        "worldpop_coverage_min_pct": 98.0,
        "min_events_after_filters": 1,
    },
    "worldpop": wp_cov,
    "acled_events": {
        "n_events_after_filters": int(len(events_wgs84)),
        "n_events_inside_admin": events_inside_n,
        "event_bounds_4326": [float(x) for x in events_bounds],
        "event_bbox_coverage_pct_of_admin": float(events_cov["coverage_pct"]),
    },
}
_write_run_metadata()

if float(wp_cov["coverage_pct"]) < 98.0:
    raise RuntimeError(
        f"WorldPop coverage below threshold for {ISO3}: {wp_cov['coverage_pct']:.3f}% < 98.000%."
    )
if int(len(events_wgs84)) < 1:
    raise RuntimeError(f"No ACLED events available for {ISO3} in selected window.")
if events_inside_n < 1:
    raise RuntimeError(f"No filtered ACLED events intersect the {ISO3} admin boundaries.")

# QC plot: ADM2 boundaries + events
fig, ax = plt.subplots(figsize=(9, 9))
admin2_gdf.boundary.plot(ax=ax, linewidth=0.6, edgecolor="grey", alpha=0.8)
for km, ms, a in [(5, 6, 0.45), (2, 4, 0.55), (1, 2, 0.70)]:
    subset = events_wgs84[events_wgs84["buffer_km"] == km]
    if not subset.empty:
        subset.plot(ax=ax, markersize=ms, alpha=a, label=f"{km} km")

ax.set_title(f"{ISO3} ACLED events ({LOOKBACK_MONTHS}m window) with ADM2 boundaries")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.legend(title="Buffer radius")

plt.tight_layout()
fig.savefig(QC_EVENTS_PNG, dpi=200)
plt.show()

update_run_metadata_artifact("qc_events", QC_EVENTS_PNG, "ACLED events with ADM2 overlays")
print("Saved QC:", QC_EVENTS_PNG)

In [ ]:
# Cell 6 — Choose metric CRS for buffering + reproject events
# Outputs:
#   - BUFFER_CRS: projected CRS (metres) used for buffering
#   - events_m: events GeoDataFrame reprojected to BUFFER_CRS

from pyproj import CRS

# Buffering must be in a projected CRS (meters), not EPSG:4326 degrees.
centroid = events_wgs84.union_all().centroid
centroid_lon, centroid_lat = centroid.x, centroid.y
utm_zone = int((centroid_lon + 180) // 6) + 1
buffer_epsg = 32600 + utm_zone if centroid_lat >= 0 else 32700 + utm_zone
BUFFER_CRS = CRS.from_epsg(buffer_epsg)

events_m = events_wgs84.to_crs(BUFFER_CRS)

print("Buffering CRS EPSG:", buffer_epsg)
print("UTM zone:", utm_zone, "Hemisphere:", "N" if centroid_lat >= 0 else "S")
print("Reprojected events bounds (meters):", events_m.total_bounds)

RUN_METADATA["buffering"] = {
    "method": "utm_from_event_centroid",
    "buffer_crs_epsg": int(buffer_epsg),
    "utm_zone": int(utm_zone),
    "hemisphere": "north" if centroid_lat >= 0 else "south",
}
_write_run_metadata()
print("Updated:", RUN_METADATA_PATH.name)

In [ ]:
# Cell 7 — Buffer events (1/2/5 km), union footprint, and QC plot
# Outputs:
#   - footprint_m / footprint_wgs84
#   - buffered_geoms_wgs84 (used for event-count rasterization on WorldPop grid)
#   - VIOL_FOOTPRINT_GPKG
#   - QC_FOOTPRINT_PNG
#   - BUFFER_STATUS_JSON progress snapshots

import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.ops import unary_union

from wia_pipelines.hazards.violence import make_progress_writer

buffers_m = events_m["buffer_km"].to_numpy(dtype=float) * 1000.0
write_progress = make_progress_writer(BUFFER_STATUS_JSON, stage="buffer_union", total=len(buffers_m))
write_progress(0, current="start")

buffered_geoms = []
chunk = max(1, len(buffers_m) // 20)
for i, (geom, dist) in enumerate(zip(events_m.geometry.values, buffers_m), start=1):
    buffered_geoms.append(geom.buffer(dist))
    if i % chunk == 0 or i == len(buffers_m):
        payload = write_progress(i, ok=i, current=f"buffered_{i}")
        print(f"Buffer progress {i}/{len(buffers_m)} ({payload['pct_complete']:.1f}%)")

# Convert buffered geometries to WGS84 for rasterization against WorldPop
buffered_geoms_wgs84 = list(gpd.GeoSeries(buffered_geoms, crs=BUFFER_CRS).to_crs("EPSG:4326").values)

# Merge overlapping buffers into one hazard footprint
footprint_geom = unary_union(buffered_geoms)
footprint_m = gpd.GeoDataFrame(
    {"run_id": [RUN_ID], "n_events": [len(events_m)]},
    geometry=[footprint_geom],
    crs=BUFFER_CRS,
)
footprint_wgs84 = footprint_m.to_crs("EPSG:4326")

# Save intermediate footprint vector
footprint_wgs84.to_file(VIOL_FOOTPRINT_GPKG, driver="GPKG")
update_run_metadata_artifact("violence_footprint", VIOL_FOOTPRINT_GPKG, "Unioned ACLED buffer footprint")
print("Wrote footprint:", VIOL_FOOTPRINT_GPKG)

# QC plot: admin + footprint + event points
fig, ax = plt.subplots(figsize=(9, 9))
admin2_gdf.boundary.plot(ax=ax, linewidth=0.6, edgecolor="grey", alpha=0.8)
footprint_wgs84.boundary.plot(ax=ax, linewidth=1.2, edgecolor="black", alpha=0.9, label="Union footprint")
for km, ms, a in [(5, 6, 0.35), (2, 4, 0.50), (1, 2, 0.65)]:
    subset = events_wgs84[events_wgs84["buffer_km"] == km]
    if not subset.empty:
        subset.plot(ax=ax, markersize=ms, alpha=a, label=f"Events ({km} km)")

ax.set_title(f"{ISO3} ACLED unioned buffer footprint ({LOOKBACK_MONTHS}m) + ADM2")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.legend(loc="lower left", fontsize=9)

plt.tight_layout()
fig.savefig(QC_FOOTPRINT_PNG, dpi=200)
plt.show()

update_run_metadata_artifact("qc_footprint", QC_FOOTPRINT_PNG, "Union footprint preview")
print("Saved QC:", QC_FOOTPRINT_PNG)

RUN_METADATA["footprint"] = {
    "footprint_gpkg": str(VIOL_FOOTPRINT_GPKG),
    "n_events": int(len(events_m)),
    "buffer_km_counts": acled_df["buffer_km"].value_counts().sort_index().to_dict(),
}
write_progress(len(buffers_m), ok=len(buffers_m), current="complete")
_write_run_metadata()
print("Updated:", RUN_METADATA_PATH.name)

In [ ]:
# Cell 8 — Rasterize buffered events to count raster, then derive binary mask + QC
# Outputs:
#   - VIOL_EVENT_COUNT_TIF (uint32, number of buffered events per pixel)
#   - VIOL_MASK_TIF (uint8, thresholded from event count)
#   - QC_MASK_PNG
#   - RASTER_STATUS_JSON progress snapshots

import numpy as np
import rasterio
from rasterio.features import rasterize
from rasterio.enums import MergeAlg
import matplotlib.pyplot as plt

from wia_pipelines.hazards.violence import make_progress_writer

MASK_THRESHOLD_EVENTS = 1

write_progress = make_progress_writer(RASTER_STATUS_JSON, stage="rasterize_event_count_and_mask", total=4)
write_progress(0, current="start")

if not buffered_geoms_wgs84:
    raise ValueError("No buffered geometries found; cannot rasterize event counts.")

# Additive rasterization in WGS84: each buffered event contributes +1 where it covers a pixel.
event_count = rasterize(
    ((geom, 1) for geom in buffered_geoms_wgs84),
    out_shape=(WP_HEIGHT, WP_WIDTH),
    transform=WP_TRANSFORM,
    fill=0,
    dtype="uint32",
    merge_alg=MergeAlg.add,
    all_touched=True,
)
write_progress(1, ok=1, current="event_count_ready")

event_count_profile = WP_PROFILE.copy()
event_count_profile.update(
    dtype="uint32",
    count=1,
    nodata=0,
    compress="deflate",
    tiled=True,
    blockxsize=512,
    blockysize=512,
)
with rasterio.open(VIOL_EVENT_COUNT_TIF, "w", **event_count_profile) as dst:
    dst.write(event_count, 1)
write_progress(2, ok=2, current="event_count_written")

mask = (event_count >= MASK_THRESHOLD_EVENTS).astype("uint8")
mask_profile = WP_PROFILE.copy()
mask_profile.update(
    dtype="uint8",
    count=1,
    nodata=0,
    compress="deflate",
    tiled=True,
    blockxsize=512,
    blockysize=512,
)
with rasterio.open(VIOL_MASK_TIF, "w", **mask_profile) as dst:
    dst.write(mask, 1)
write_progress(3, ok=3, current="mask_written")

print("Wrote event count raster:", VIOL_EVENT_COUNT_TIF)
print("Wrote mask raster:", VIOL_MASK_TIF)
print(f"Max event count per pixel: {int(event_count.max()):,}")
print(
    f"Affected pixels (>= {MASK_THRESHOLD_EVENTS}): {int(mask.sum()):,} / {mask.size:,} ({mask.mean() * 100:.2f}%)"
)

# QC overlay: WorldPop base + binary mask + ADM2 boundaries
with rasterio.open(WORLDPOP_PATH) as src:
    wp = src.read(1).astype("float32")
    nodata = src.nodata
    b = src.bounds
if nodata is not None:
    wp[wp == nodata] = np.nan

extent = (b.left, b.right, b.bottom, b.top)
fig, ax = plt.subplots(figsize=(9, 9))
ax.imshow(wp, interpolation="nearest", extent=extent, origin="upper", zorder=1)
ax.imshow(
    np.where(mask == 1, 1, np.nan),
    interpolation="nearest",
    extent=extent,
    origin="upper",
    alpha=0.35,
    zorder=2,
)
admin2_gdf.boundary.plot(ax=ax, linewidth=0.6, edgecolor="grey", alpha=0.8, zorder=3)

ax.set_title(f"{ISO3} violence hazard impact mask (count >= {MASK_THRESHOLD_EVENTS}, {LOOKBACK_MONTHS}m)")
ax.set_xlim(b.left, b.right)
ax.set_ylim(b.bottom, b.top)
ax.axis("off")

plt.tight_layout()
fig.savefig(QC_MASK_PNG, dpi=200)
plt.show()

update_run_metadata_artifact("event_count", VIOL_EVENT_COUNT_TIF, "Buffered ACLED event count raster")
update_run_metadata_artifact("hazard_mask", VIOL_MASK_TIF, "Binary hazard mask from event count threshold")
update_run_metadata_artifact("qc_mask", QC_MASK_PNG, "Mask overlay on WorldPop")

RUN_METADATA["hazard_intensity"] = {
    "event_count_tif": str(VIOL_EVENT_COUNT_TIF),
    "dtype": "uint32",
    "max_event_count": int(event_count.max()),
    "all_touched": True,
}
RUN_METADATA["hazard_mask"] = {
    "mask_tif": str(VIOL_MASK_TIF),
    "dtype": "uint8",
    "threshold_metric": "event_count",
    "threshold_value": int(MASK_THRESHOLD_EVENTS),
    "affected_pixels": int(mask.sum()),
    "total_pixels": int(mask.size),
}

write_progress(4, ok=4, current="complete")
_write_run_metadata()
print("Saved QC:", QC_MASK_PNG)

In [ ]:
# Cell 9 — Population affected and pop-weighted event count rasters
# Outputs:
#   - VIOL_POP_AFFECTED_TIF (float32): WorldPop x binary violence mask
#   - VIOL_POP_WEIGHTED_EVENT_COUNT_TIF (float32): WorldPop x event_count
#   - national totals and population-weighted mean event count

import numpy as np
import rasterio
from affine import Affine

from wia_pipelines.hazards.violence import make_progress_writer

write_progress = make_progress_writer(RASTER_STATUS_JSON, stage="population_impact_rasters", total=4)
write_progress(0, current="start")

# Load WorldPop reference data
with rasterio.open(WORLDPOP_PATH) as src:
    wp = src.read(1).astype("float64")
    wp_nodata = src.nodata
    wp_crs = src.crs
    wp_transform = src.transform
    wp_shape = (src.height, src.width)

# Load thresholded binary mask
with rasterio.open(VIOL_MASK_TIF) as src:
    mask_arr = src.read(1).astype("uint8")
    mask_crs = src.crs
    mask_transform = src.transform
    mask_shape = (src.height, src.width)

# Load event count raster
with rasterio.open(VIOL_EVENT_COUNT_TIF) as src:
    event_count_arr = src.read(1).astype("float64")
    count_crs = src.crs
    count_transform = src.transform
    count_shape = (src.height, src.width)

# Enforce grid alignment between WorldPop, binary mask, and event-count raster
same_mask = (
    wp_crs == mask_crs
    and wp_shape == mask_shape
    and isinstance(mask_transform, Affine)
    and isinstance(wp_transform, Affine)
    and mask_transform.almost_equals(wp_transform, precision=1e-9)
)
same_count = (
    wp_crs == count_crs
    and wp_shape == count_shape
    and isinstance(count_transform, Affine)
    and isinstance(wp_transform, Affine)
    and count_transform.almost_equals(wp_transform, precision=1e-9)
)
if not same_mask:
    raise ValueError("Violence mask raster is not aligned to WorldPop grid.")
if not same_count:
    raise ValueError("Violence event-count raster is not aligned to WorldPop grid.")

if wp_nodata is not None:
    wp = np.where(wp == wp_nodata, 0.0, wp)

affected_pop = wp * mask_arr
pop_weighted_event_count = wp * event_count_arr
write_progress(1, ok=1, current="impact_arrays_computed")

out_profile = WP_PROFILE.copy()
out_profile.update(
    dtype="float32",
    count=1,
    nodata=0,
    compress="deflate",
    tiled=True,
    blockxsize=512,
    blockysize=512,
)

with rasterio.open(VIOL_POP_AFFECTED_TIF, "w", **out_profile) as dst:
    dst.write(affected_pop.astype("float32"), 1)
write_progress(2, ok=2, current="pop_affected_written")

with rasterio.open(VIOL_POP_WEIGHTED_EVENT_COUNT_TIF, "w", **out_profile) as dst:
    dst.write(pop_weighted_event_count.astype("float32"), 1)
write_progress(3, ok=3, current="pop_weighted_event_count_written")

total_pop = float(wp.sum())
pop_affected = float(affected_pop.sum())
pct_affected = (pop_affected / total_pop * 100.0) if total_pop > 0 else np.nan
national_pop_weighted_mean_event_count = (
    float(pop_weighted_event_count.sum()) / total_pop if total_pop > 0 else np.nan
)

print("Wrote affected population raster:", VIOL_POP_AFFECTED_TIF)
print("Wrote pop-weighted event-count raster:", VIOL_POP_WEIGHTED_EVENT_COUNT_TIF)
print(f"Total population (raster extent): {total_pop:,.0f}")
print(f"Affected population (>=1 event): {pop_affected:,.0f}")
print(f"% affected: {pct_affected:.2f}%")
print(f"Population-weighted mean event count (national): {national_pop_weighted_mean_event_count:.4f}")

update_run_metadata_artifact(
    "pop_affected_raster", VIOL_POP_AFFECTED_TIF, "WorldPop multiplied by thresholded violence mask"
)
update_run_metadata_artifact(
    "pop_weighted_event_count_raster",
    VIOL_POP_WEIGHTED_EVENT_COUNT_TIF,
    "WorldPop multiplied by violence event count raster",
)
RUN_METADATA["population_impact"] = {
    "affected_pop_tif": str(VIOL_POP_AFFECTED_TIF),
    "pop_weighted_event_count_tif": str(VIOL_POP_WEIGHTED_EVENT_COUNT_TIF),
    "total_population": total_pop,
    "affected_population": pop_affected,
    "pct_population_affected": pct_affected,
    "pop_weighted_mean_event_count": national_pop_weighted_mean_event_count,
}
write_progress(4, ok=4, current="complete")
_write_run_metadata()
print("Updated:", RUN_METADATA_PATH.name)

In [ ]:
# Cell 10 — ADM2 zonal statistics (total pop, affected pop, % affected, pop-weighted mean event count)
# Outputs:
#   - adm2_stats_df
#   - ADM2_STATS_CSV
#   - ZONAL_STATUS_JSON progress snapshots

from rasterstats import zonal_stats
import pandas as pd
import numpy as np

from wia_pipelines.hazards.violence import make_progress_writer

write_progress = make_progress_writer(ZONAL_STATUS_JSON, stage="adm2_zonal_stats", total=5)
write_progress(0, current="start")

# Ensure admin geometry CRS matches raster CRS (EPSG:4326 in this pipeline)
admin2_zs = admin2_gdf.to_crs("EPSG:4326") if str(admin2_gdf.crs).upper() != "EPSG:4326" else admin2_gdf

zs_total = zonal_stats(
    admin2_zs,
    str(WORLDPOP_PATH),
    stats=["sum"],
    nodata=WP_NODATA,
    all_touched=True,
)
write_progress(1, ok=1, current="total_population_zonal_done")

zs_aff = zonal_stats(
    admin2_zs,
    str(VIOL_POP_AFFECTED_TIF),
    stats=["sum"],
    nodata=0,
    all_touched=True,
)
write_progress(2, ok=2, current="affected_population_zonal_done")

zs_weighted_count = zonal_stats(
    admin2_zs,
    str(VIOL_POP_WEIGHTED_EVENT_COUNT_TIF),
    stats=["sum"],
    nodata=0,
    all_touched=True,
)
write_progress(3, ok=3, current="weighted_event_count_zonal_done")

pop_total = np.array([d["sum"] if d["sum"] is not None else 0.0 for d in zs_total], dtype="float64")
pop_aff = np.array([d["sum"] if d["sum"] is not None else 0.0 for d in zs_aff], dtype="float64")
pop_weighted_count_sum = np.array(
    [d["sum"] if d["sum"] is not None else 0.0 for d in zs_weighted_count], dtype="float64"
)
pct_aff = np.where(pop_total > 0, pop_aff / pop_total * 100.0, np.nan)
pop_weighted_mean_event_count = np.where(pop_total > 0, pop_weighted_count_sum / pop_total, np.nan)

adm2_stats_df = pd.DataFrame(
    {
        "iso3": ISO3,
        "adm2_pcode": admin2_zs["adm2_pcode"].astype(str).values,
        "pop_total": pop_total,
        "pop_affected": pop_aff,
        "pct_affected": pct_aff,
        "pop_weighted_event_count_sum": pop_weighted_count_sum,
        "pop_weighted_mean_event_count": pop_weighted_mean_event_count,
    }
)

adm2_stats_df.to_csv(ADM2_STATS_CSV, index=False)
write_progress(4, ok=4, current="adm2_table_written")

print("ADM2 rows:", len(adm2_stats_df))
print("Affected pop (sum ADM2):", f"{adm2_stats_df['pop_affected'].sum():,.0f}")
print("Total pop (sum ADM2):", f"{adm2_stats_df['pop_total'].sum():,.0f}")
print(
    "Pop-weighted mean event count (national from ADM2):",
    f"{(adm2_stats_df['pop_weighted_event_count_sum'].sum() / max(adm2_stats_df['pop_total'].sum(), 1.0)):.4f}",
)
print("Wrote:", ADM2_STATS_CSV)

update_run_metadata_artifact("adm2_stats", ADM2_STATS_CSV, "ADM2 population affected summary")
RUN_METADATA["adm2_stats"] = {
    "adm2_stats_csv": str(ADM2_STATS_CSV),
    "n_adm2": int(len(adm2_stats_df)),
    "adm2_pop_total_sum": float(adm2_stats_df["pop_total"].sum()),
    "adm2_pop_affected_sum": float(adm2_stats_df["pop_affected"].sum()),
    "adm2_pop_weighted_event_count_sum": float(adm2_stats_df["pop_weighted_event_count_sum"].sum()),
    "adm2_pop_weighted_mean_event_count": float(
        adm2_stats_df["pop_weighted_event_count_sum"].sum() / max(adm2_stats_df["pop_total"].sum(), 1.0)
    ),
}
write_progress(5, ok=5, current="complete")
_write_run_metadata()
print("Updated:", RUN_METADATA_PATH.name)

display(adm2_stats_df.head())

In [ ]:
# Cell 11 — Final QC: join ADM2 stats + choropleth map + distribution checks
# Outputs:
#   - admin2_qc_gdf
#   - QC_ADM2_MAP_PNG

import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt

admin2_qc_gdf = admin2_gdf.merge(adm2_stats_df, on=["iso3", "adm2_pcode"], how="left")
admin2_qc_gdf["pop_total"] = admin2_qc_gdf["pop_total"].fillna(0.0)
admin2_qc_gdf["pop_affected"] = admin2_qc_gdf["pop_affected"].fillna(0.0)
admin2_qc_gdf["pct_affected"] = np.where(
    admin2_qc_gdf["pop_total"] > 0,
    admin2_qc_gdf["pop_affected"] / admin2_qc_gdf["pop_total"] * 100.0,
    np.nan,
)

pct = admin2_qc_gdf["pct_affected"].dropna()
print("ADM2 % affected summary")
print(f"  ADM2 units: {len(admin2_qc_gdf):,}")
print(f"  Non-null pct: {len(pct):,}")
print(f"  Min/Median/Max: {pct.min():.2f}% / {pct.median():.2f}% / {pct.max():.2f}%")
print("  Percentiles (5, 25, 75, 95):", ", ".join([f"{p:.2f}%" for p in np.percentile(pct, [5, 25, 75, 95])]))

fig, ax = plt.subplots(figsize=(10, 8))
admin2_qc_gdf.plot(
    column="pct_affected",
    ax=ax,
    legend=True,
    linewidth=0.2,
    edgecolor="black",
    missing_kwds={"color": "lightgrey", "label": "No data"},
)
ax.set_title(
    f"{ISO3} - ADM2 % population affected by >=1 violent event ({WINDOW_START_DATE} -> {WINDOW_END_DATE})"
)
ax.axis("off")

plt.tight_layout()
fig.savefig(QC_ADM2_MAP_PNG, dpi=200)
plt.show()

update_run_metadata_artifact("qc_adm2", QC_ADM2_MAP_PNG, "ADM2 percent affected choropleth")
RUN_METADATA["qc_outputs"] = {
    "qc_events_png": str(QC_EVENTS_PNG),
    "qc_footprint_png": str(QC_FOOTPRINT_PNG),
    "qc_mask_png": str(QC_MASK_PNG),
    "qc_adm2_map_png": str(QC_ADM2_MAP_PNG),
}
_write_run_metadata()

print("Saved QC:", QC_ADM2_MAP_PNG)
print("Updated:", RUN_METADATA_PATH.name)
print("Next: run scripts/violence_postrun.py --run-dir", RUN_DIR)